In [16]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [17]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars = list(value_dict.values())
    expectation_value = estimator.run(ansatz, qubit_op, pars).result().values
    return np.real(expectation_value[0])  # Ensure it returns a scalar

In [18]:
from mealpy import FloatVar, SMA
import numpy as np

def objective_function(solution):
    return np.sum(solution**2)

def rastrigin_function(X):
    A = 10
    return A * len(X) + sum([(x**2 - A * np.cos(2 * np.pi * x)) for x in X])

problem = {
    "obj_func": rastrigin_function,
    "bounds": FloatVar(lb=(-100., )*30, ub=(100., )*30),
    "minmax": "min",
    "log_to": None,
}

## Run the algorithm
model = SMA.OriginalSMA(epoch=100, pop_size=50, pr=0.03)
g_best = model.solve(problem)
print(f"Best solution: {g_best.solution}, Best fitness: {g_best.target.fitness}")

Best solution: [ 3.67694985e-10 -4.49460147e-12 -1.21198545e-11 -3.10560953e-13
 -8.18549640e-12 -2.10909816e-11  5.20411399e-11 -3.18497529e-13
 -5.51656615e-11  5.39883563e-11 -7.56159568e-12  3.91986091e-13
 -2.05034931e-13 -9.22309575e-11  4.94608662e-13  3.41757889e-12
 -1.52314349e-10  2.17118322e-12  3.36739962e-12 -1.60718302e-10
 -9.74506903e-10  2.60942808e-11  1.37472769e-11  1.00510339e-12
 -5.56901032e-11  2.64505863e-11  3.36090912e-11 -1.57309367e-10
  6.15655325e-11  3.47232796e-15], Best fitness: 0.0


In [ ]:
min_qubits = 5
max_qubits=6
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                backend_options={ # method chosen automatically to match options
                    "coupling_map": coupling_map,
                },
                run_options={"seed": 42, "shots": 1024},
                transpile_options={"seed_transpiler": 42},
            )
    # Example usage
    from mealpy import FloatVar, SMA
    import numpy as np

    problem = {
        "obj_func": evaluate_expectation,
        "bounds": FloatVar(lb=(-100., )*dimension, ub=(100., )*dimension),
        "minmax": "min",
        "log_to": None,
    }

    ## Run the algorithm
    model = SMA.OriginalSMA(epoch=300, pop_size=100, pr=0.03)
    g_best = model.solve(problem)
    print(f"Best solution: {g_best.solution}, Best fitness: {g_best.target.fitness}")

ansatz_num_parameters= 20
Best solution: [-70.09329397 -37.35265903  62.32468475 -36.43943479 -41.57066589
   1.1058312   68.13527159  -1.13201056  73.2712252   50.95347172
  89.32733097   6.6081268   51.56795602  74.43537549  23.29639797
  70.96883245 -98.13285327  77.01615677 -49.55057095 -67.44214343], Best fitness: -3.84375


In [ ]:
from mealpy.optimizer import Optimizer

In [27]:
#!/usr/bin/env python
# Created by "Thieu" at 12:51, 18/03/2020 ----------%
#       Email: nguyenthieu2102@gmail.com            %
#       Github: https://github.com/thieu1995        %
# --------------------------------------------------%

import numpy as np
from mealpy.optimizer import Optimizer


class OriginalBBOA(Optimizer):
    """
    The original version of: Brown-Bear Optimization Algorithm (BBOA)

    Links:
        1. https://www.mathworks.com/matlabcentral/fileexchange/125490-brown-bear-optimization-algorithm

    Examples
    ~~~~~~~~
     import numpy as np
     from mealpy import FloatVar, BBOA
    
     def objective_function(solution):
         return np.sum(solution**2)
    
     problem_dict = {
         "bounds": FloatVar(lb=(-10.,) * 30, ub=(10.,) * 30, name="delta"),
         "minmax": "min",
         "obj_func": objective_function
     }
    
     model = BBOA.OriginalBBOA(epoch=1000, pop_size=50)
     g_best = model.solve(problem_dict)
     print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
     print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

    References
    ~~~~~~~~~~
    [1] Prakash, T., Singh, P. P., Singh, V. P., & Singh, S. N. (2023). A Novel Brown-bear Optimization
    Algorithm for Solving Economic Dispatch Problem. In Advanced Control & Optimization Paradigms for
    Energy System Operation and Management (pp. 137-164). River Publishers.
    """

    def __init__(self, epoch: int = 10000, pop_size: int = 100, **kwargs: object) -> None:
        """
        Args:
            epoch (int): maximum number of iterations, default = 10000
            pop_size (int): number of population size, default = 100
        """
        super().__init__(**kwargs)
        self.epoch = self.validator.check_int("epoch", epoch, [1, 100000])
        self.pop_size = self.validator.check_int("pop_size", pop_size, [5, 10000])
        self.set_parameters(["epoch", "pop_size"])
        self.sort_flag = False

    def evolve(self, epoch):
        """
        The main operations (equations) of algorithm. Inherit from Optimizer class

        Args:
            epoch (int): The current iteration
        """
        pp = epoch / self.epoch

        ## Pedal marking behaviour
        pop_new = []
        for idx in range(0, self.pop_size):
            if pp <= epoch/3:           # Gait while walking
                pos_new = self.pop[idx].solution + (-pp * self.generator.random(self.problem.n_dims) * self.pop[idx].solution)
            elif epoch/3 < pp <= 2*epoch/3:     # Careful Stepping
                qq = pp * self.generator.random(self.problem.n_dims)
                pos_new = self.pop[idx].solution + (qq * (self.g_best.solution - self.generator.integers(1, 3) * self.g_worst.solution))
            else:
                ww = 2 * pp * np.pi * self.generator.random(self.problem.n_dims)
                pos_new = self.pop[idx].solution + (ww*self.g_best.solution - np.abs(self.pop[idx].solution)) - (ww*self.g_worst.solution - np.abs(self.pop[idx].solution))
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop_new.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)

        ## Sniffing of pedal marks
        pop_new = []
        for idx in range(0, self.pop_size):
            kk = self.generator.choice(list(set(range(0, self.pop_size)) - {idx}))
            if self.compare_target(self.pop[idx].target, self.pop[kk].target, self.problem.minmax):
                pos_new = self.pop[idx].solution + self.generator.random() * (self.pop[idx].solution - self.pop[kk].solution)
            else:
                pos_new = self.pop[idx].solution + self.generator.random() * (self.pop[kk].solution - self.pop[idx].solution)
            pos_new = self.correct_solution(pos_new)
            agent = self.generate_empty_agent(pos_new)
            pop_new.append(agent)
            if self.mode not in self.AVAILABLE_MODES:
                agent.target = self.get_target(pos_new)
                self.pop[idx] = self.get_better_agent(agent, self.pop[idx], self.problem.minmax)
        if self.mode in self.AVAILABLE_MODES:
            pop_new = self.update_target_for_population(pop_new)
            self.pop = self.greedy_selection_population(self.pop, pop_new, self.problem.minmax)

In [29]:
import numpy as np

def objective_function(solution):
    return np.sum(solution**2)

problem_dict = {
"bounds": FloatVar(lb=(-10.,) * dimension, ub=(10.,) * dimension, name="delta"),
"minmax": "min",
"obj_func": evaluate_expectation
}

model = OriginalBBOA(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 01:25:03 PM, INFO, __main__.OriginalBBOA: Solving single objective optimization problem.


2024/07/10 01:25:04 PM, INFO, __main__.OriginalBBOA: >>>Problem: P, Epoch: 1, Current best: -1.5, Global best: -1.5, Runtime: 1.30972 seconds
2024/07/10 01:25:05 PM, INFO, __main__.OriginalBBOA: >>>Problem: P, Epoch: 2, Current best: -1.5, Global best: -1.5, Runtime: 0.38617 seconds
2024/07/10 01:25:05 PM, INFO, __main__.OriginalBBOA: >>>Problem: P, Epoch: 3, Current best: -1.5, Global best: -1.5, Runtime: 0.34170 seconds
2024/07/10 01:25:06 PM, INFO, __main__.OriginalBBOA: >>>Problem: P, Epoch: 4, Current best: -1.5, Global best: -1.5, Runtime: 0.34242 seconds
2024/07/10 01:25:06 PM, INFO, __main__.OriginalBBOA: >>>Problem: P, Epoch: 5, Current best: -1.5, Global best: -1.5, Runtime: 0.32144 seconds
2024/07/10 01:25:06 PM, INFO, __main__.OriginalBBOA: >>>Problem: P, Epoch: 6, Current best: -1.5, Global best: -1.5, Runtime: 0.33991 seconds
2024/07/10 01:25:07 PM, INFO, __main__.OriginalBBOA: >>>Problem: P, Epoch: 7, Current best: -1.5, Global best: -1.5, Runtime: 0.31699 seconds
2024/0

Solution: [ 5.88237792 -5.36274604 -7.25979646 -4.65730659  0.11359949  4.27570826
  7.26641213  3.91050281  0.83308432  7.22336079  1.53294653  7.42218302
  4.12373366  1.58927418 -4.55523733 -4.44898163 -0.25720375 -1.81135767
 -0.943045   -4.26572031], Fitness: -3.15625
Solution: [ 5.88237792 -5.36274604 -7.25979646 -4.65730659  0.11359949  4.27570826
  7.26641213  3.91050281  0.83308432  7.22336079  1.53294653  7.42218302
  4.12373366  1.58927418 -4.55523733 -4.44898163 -0.25720375 -1.81135767
 -0.943045   -4.26572031], Fitness: -3.15625


In [34]:
import numpy as np
from mealpy import FloatVar, BBO

problem_dict = {
     "bounds": FloatVar(lb=(-1000.,) * dimension, ub=(1000.,) * dimension, name="delta"),
     "minmax": "min",
     "obj_func": evaluate_expectation
}

model = BBO.OriginalBBO(epoch=1000, pop_size=100, p_m=0.01, n_elites=2)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 01:39:42 PM, INFO, mealpy.bio_based.BBO.OriginalBBO: Solving single objective optimization problem.
2024/07/10 01:39:43 PM, INFO, mealpy.bio_based.BBO.OriginalBBO: >>>Problem: P, Epoch: 1, Current best: -1.5625, Global best: -1.5625, Runtime: 0.39413 seconds
2024/07/10 01:39:43 PM, INFO, mealpy.bio_based.BBO.OriginalBBO: >>>Problem: P, Epoch: 2, Current best: -1.625, Global best: -1.625, Runtime: 0.40000 seconds
2024/07/10 01:39:44 PM, INFO, mealpy.bio_based.BBO.OriginalBBO: >>>Problem: P, Epoch: 3, Current best: -1.75, Global best: -1.75, Runtime: 0.40285 seconds
2024/07/10 01:39:44 PM, INFO, mealpy.bio_based.BBO.OriginalBBO: >>>Problem: P, Epoch: 4, Current best: -1.75, Global best: -1.75, Runtime: 0.34835 seconds
2024/07/10 01:39:44 PM, INFO, mealpy.bio_based.BBO.OriginalBBO: >>>Problem: P, Epoch: 5, Current best: -1.75, Global best: -1.75, Runtime: 0.37301 seconds
2024/07/10 01:39:45 PM, INFO, mealpy.bio_based.BBO.OriginalBBO: >>>Problem: P, Epoch: 6, Current best: -1.75

KeyboardInterrupt: 

In [37]:
 import numpy as np
from mealpy import FloatVar, BMO


problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = BMO.OriginalBMO(epoch=1000, pop_size=200, pl = 4)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 01:45:32 PM, INFO, mealpy.bio_based.BMO.OriginalBMO: Solving single objective optimization problem.
2024/07/10 01:45:33 PM, INFO, mealpy.bio_based.BMO.OriginalBMO: >>>Problem: P, Epoch: 1, Current best: -1.96875, Global best: -1.96875, Runtime: 0.64452 seconds
2024/07/10 01:45:34 PM, INFO, mealpy.bio_based.BMO.OriginalBMO: >>>Problem: P, Epoch: 2, Current best: -2.25, Global best: -2.25, Runtime: 0.64590 seconds
2024/07/10 01:45:34 PM, INFO, mealpy.bio_based.BMO.OriginalBMO: >>>Problem: P, Epoch: 3, Current best: -1.625, Global best: -2.25, Runtime: 0.62399 seconds
2024/07/10 01:45:35 PM, INFO, mealpy.bio_based.BMO.OriginalBMO: >>>Problem: P, Epoch: 4, Current best: -1.875, Global best: -2.25, Runtime: 0.57204 seconds
2024/07/10 01:45:35 PM, INFO, mealpy.bio_based.BMO.OriginalBMO: >>>Problem: P, Epoch: 5, Current best: -1.3125, Global best: -2.25, Runtime: 0.55551 seconds
2024/07/10 01:45:36 PM, INFO, mealpy.bio_based.BMO.OriginalBMO: >>>Problem: P, Epoch: 6, Current best: -

KeyboardInterrupt: 

In [38]:
from mealpy import FloatVar, EOA

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = EOA.OriginalEOA(epoch=1000, pop_size=50, p_c = 0.9, p_m = 0.01, n_best = 2, alpha = 0.98, beta = 0.9, gama = 0.9)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 01:47:44 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: Solving single objective optimization problem.
2024/07/10 01:47:45 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 1, Current best: -2.25, Global best: -2.25, Runtime: 0.36652 seconds
2024/07/10 01:47:45 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 2, Current best: -2.25, Global best: -2.25, Runtime: 0.39003 seconds
2024/07/10 01:47:45 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 3, Current best: -2.25, Global best: -2.25, Runtime: 0.36675 seconds
2024/07/10 01:47:46 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 4, Current best: -2.25, Global best: -2.25, Runtime: 0.37386 seconds
2024/07/10 01:47:46 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 5, Current best: -2.25, Global best: -2.25, Runtime: 0.32412 seconds
2024/07/10 01:47:46 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 6, Current best: -2.25, Glob

Solution: [-22.76284446  94.61736553  39.72056733 -38.10990895 -28.48117101
 -11.64327403  64.16644258 -46.04294503  25.68664425 -18.42795402
 -26.63916689  41.80633835  17.04739389  13.78586559  10.62930754
 -20.29676705  59.1262938  -14.05072736  35.92399364  -7.44594837], Fitness: -3.8125
Solution: [-22.76284446  94.61736553  39.72056733 -38.10990895 -28.48117101
 -11.64327403  64.16644258 -46.04294503  25.68664425 -18.42795402
 -26.63916689  41.80633835  17.04739389  13.78586559  10.62930754
 -20.29676705  59.1262938  -14.05072736  35.92399364  -7.44594837], Fitness: -3.8125


In [40]:
from mealpy import FloatVar, EOA

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = EOA.OriginalEOA(epoch=1000, pop_size=200, seed_min = 3, seed_max = 9, exponent = 3, sigma_start = 0.6, sigma_end = 0.01)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:06:44 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: Solving single objective optimization problem.
2024/07/10 03:06:46 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 1, Current best: -2.1875, Global best: -2.1875, Runtime: 1.67553 seconds
2024/07/10 03:06:48 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 2, Current best: -2.28125, Global best: -2.28125, Runtime: 1.56792 seconds
2024/07/10 03:06:49 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 3, Current best: -2.28125, Global best: -2.28125, Runtime: 1.66635 seconds
2024/07/10 03:06:51 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 4, Current best: -2.28125, Global best: -2.28125, Runtime: 1.53478 seconds
2024/07/10 03:06:52 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 5, Current best: -2.28125, Global best: -2.28125, Runtime: 1.40261 seconds
2024/07/10 03:06:54 PM, INFO, mealpy.bio_based.EOA.OriginalEOA: >>>Problem: P, Epoch: 

KeyboardInterrupt: 

In [44]:
from mealpy import FloatVar, SBO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = SBO.DevSBO(epoch=1000, pop_size=200, alpha = 0.7, p_m=0.05, psw = 0.02)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:12:28 PM, INFO, mealpy.bio_based.SBO.DevSBO: Solving single objective optimization problem.
2024/07/10 03:12:30 PM, INFO, mealpy.bio_based.SBO.DevSBO: >>>Problem: P, Epoch: 1, Current best: -1.625, Global best: -1.625, Runtime: 0.94858 seconds
2024/07/10 03:12:30 PM, INFO, mealpy.bio_based.SBO.DevSBO: >>>Problem: P, Epoch: 2, Current best: -1.625, Global best: -1.625, Runtime: 0.75168 seconds
2024/07/10 03:12:31 PM, INFO, mealpy.bio_based.SBO.DevSBO: >>>Problem: P, Epoch: 3, Current best: -2.375, Global best: -2.375, Runtime: 0.76546 seconds
2024/07/10 03:12:32 PM, INFO, mealpy.bio_based.SBO.DevSBO: >>>Problem: P, Epoch: 4, Current best: -2.375, Global best: -2.375, Runtime: 0.74934 seconds
2024/07/10 03:12:33 PM, INFO, mealpy.bio_based.SBO.DevSBO: >>>Problem: P, Epoch: 5, Current best: -2.375, Global best: -2.375, Runtime: 0.75120 seconds
2024/07/10 03:12:33 PM, INFO, mealpy.bio_based.SBO.DevSBO: >>>Problem: P, Epoch: 6, Current best: -2.375, Global best: -2.375, Runtime

KeyboardInterrupt: 

In [50]:
from mealpy import FloatVar, SOA

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = SOA.DevSOA(epoch=1000, pop_size=75, fc = 4)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:20:47 PM, INFO, mealpy.bio_based.SOA.DevSOA: Solving single objective optimization problem.
2024/07/10 03:20:48 PM, INFO, mealpy.bio_based.SOA.DevSOA: >>>Problem: P, Epoch: 1, Current best: -1.5625, Global best: -1.5625, Runtime: 0.30146 seconds
2024/07/10 03:20:48 PM, INFO, mealpy.bio_based.SOA.DevSOA: >>>Problem: P, Epoch: 2, Current best: -1.5625, Global best: -1.5625, Runtime: 0.35476 seconds
2024/07/10 03:20:48 PM, INFO, mealpy.bio_based.SOA.DevSOA: >>>Problem: P, Epoch: 3, Current best: -1.5625, Global best: -1.5625, Runtime: 0.30522 seconds
2024/07/10 03:20:49 PM, INFO, mealpy.bio_based.SOA.DevSOA: >>>Problem: P, Epoch: 4, Current best: -1.5625, Global best: -1.5625, Runtime: 0.29379 seconds
2024/07/10 03:20:49 PM, INFO, mealpy.bio_based.SOA.DevSOA: >>>Problem: P, Epoch: 5, Current best: -1.5625, Global best: -1.5625, Runtime: 0.28533 seconds
2024/07/10 03:20:49 PM, INFO, mealpy.bio_based.SOA.DevSOA: >>>Problem: P, Epoch: 6, Current best: -1.5625, Global best: -1.5

KeyboardInterrupt: 

In [51]:
from mealpy import FloatVar, SOS

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = SOS.OriginalSOS(epoch=1000, pop_size=50)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:23:36 PM, INFO, mealpy.bio_based.SOS.OriginalSOS: Solving single objective optimization problem.
2024/07/10 03:23:37 PM, INFO, mealpy.bio_based.SOS.OriginalSOS: >>>Problem: P, Epoch: 1, Current best: -1.59375, Global best: -1.59375, Runtime: 0.59970 seconds
2024/07/10 03:23:38 PM, INFO, mealpy.bio_based.SOS.OriginalSOS: >>>Problem: P, Epoch: 2, Current best: -1.8125, Global best: -1.8125, Runtime: 0.61177 seconds
2024/07/10 03:23:38 PM, INFO, mealpy.bio_based.SOS.OriginalSOS: >>>Problem: P, Epoch: 3, Current best: -2.4375, Global best: -2.4375, Runtime: 0.64799 seconds
2024/07/10 03:23:39 PM, INFO, mealpy.bio_based.SOS.OriginalSOS: >>>Problem: P, Epoch: 4, Current best: -2.75, Global best: -2.75, Runtime: 0.71968 seconds
2024/07/10 03:23:40 PM, INFO, mealpy.bio_based.SOS.OriginalSOS: >>>Problem: P, Epoch: 5, Current best: -2.75, Global best: -2.75, Runtime: 0.69369 seconds
2024/07/10 03:23:40 PM, INFO, mealpy.bio_based.SOS.OriginalSOS: >>>Problem: P, Epoch: 6, Current bes

KeyboardInterrupt: 

In [54]:
from mealpy import FloatVar, TPO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = TPO.DevTPO(epoch=1000, pop_size=50, alpha = 0.3, beta = 50., theta = 0.9)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:32:57 PM, INFO, mealpy.bio_based.TPO.DevTPO: Solving single objective optimization problem.
2024/07/10 03:33:00 PM, INFO, mealpy.bio_based.TPO.DevTPO: >>>Problem: P, Epoch: 1, Current best: -2.125, Global best: -2.125, Runtime: 1.41540 seconds
2024/07/10 03:33:01 PM, INFO, mealpy.bio_based.TPO.DevTPO: >>>Problem: P, Epoch: 2, Current best: -2.125, Global best: -2.125, Runtime: 1.40761 seconds
2024/07/10 03:33:03 PM, INFO, mealpy.bio_based.TPO.DevTPO: >>>Problem: P, Epoch: 3, Current best: -2.125, Global best: -2.125, Runtime: 1.75132 seconds
2024/07/10 03:33:04 PM, INFO, mealpy.bio_based.TPO.DevTPO: >>>Problem: P, Epoch: 4, Current best: -2.125, Global best: -2.125, Runtime: 1.27321 seconds
2024/07/10 03:33:05 PM, INFO, mealpy.bio_based.TPO.DevTPO: >>>Problem: P, Epoch: 5, Current best: -2.125, Global best: -2.125, Runtime: 1.32014 seconds
2024/07/10 03:33:07 PM, INFO, mealpy.bio_based.TPO.DevTPO: >>>Problem: P, Epoch: 6, Current best: -2.125, Global best: -2.125, Runtime

KeyboardInterrupt: 

In [56]:
from mealpy import FloatVar, TSA

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = TSA.OriginalTSA(epoch=1000, pop_size=200)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:38:47 PM, INFO, mealpy.bio_based.TSA.OriginalTSA: Solving single objective optimization problem.
2024/07/10 03:38:48 PM, INFO, mealpy.bio_based.TSA.OriginalTSA: >>>Problem: P, Epoch: 1, Current best: -1.6875, Global best: -1.71875, Runtime: 0.77474 seconds
2024/07/10 03:38:49 PM, INFO, mealpy.bio_based.TSA.OriginalTSA: >>>Problem: P, Epoch: 2, Current best: -1.65625, Global best: -1.71875, Runtime: 0.64224 seconds
2024/07/10 03:38:49 PM, INFO, mealpy.bio_based.TSA.OriginalTSA: >>>Problem: P, Epoch: 3, Current best: -1.53125, Global best: -1.71875, Runtime: 0.60930 seconds
2024/07/10 03:38:50 PM, INFO, mealpy.bio_based.TSA.OriginalTSA: >>>Problem: P, Epoch: 4, Current best: -1.15625, Global best: -1.71875, Runtime: 0.77514 seconds
2024/07/10 03:38:51 PM, INFO, mealpy.bio_based.TSA.OriginalTSA: >>>Problem: P, Epoch: 5, Current best: -1.4375, Global best: -1.71875, Runtime: 0.82865 seconds
2024/07/10 03:38:51 PM, INFO, mealpy.bio_based.TSA.OriginalTSA: >>>Problem: P, Epoch: 

KeyboardInterrupt: 

In [59]:
from mealpy import FloatVar, VCS

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = VCS.DevVCS(epoch=1000, pop_size=50, lamda = 0.5, sigma = 0.3)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:49:04 PM, INFO, mealpy.bio_based.VCS.DevVCS: Solving single objective optimization problem.
2024/07/10 03:49:05 PM, INFO, mealpy.bio_based.VCS.DevVCS: >>>Problem: P, Epoch: 1, Current best: -1.40625, Global best: -1.40625, Runtime: 0.89934 seconds
2024/07/10 03:49:06 PM, INFO, mealpy.bio_based.VCS.DevVCS: >>>Problem: P, Epoch: 2, Current best: -1.40625, Global best: -1.40625, Runtime: 0.53605 seconds
2024/07/10 03:49:06 PM, INFO, mealpy.bio_based.VCS.DevVCS: >>>Problem: P, Epoch: 3, Current best: -1.40625, Global best: -1.40625, Runtime: 0.54109 seconds
2024/07/10 03:49:07 PM, INFO, mealpy.bio_based.VCS.DevVCS: >>>Problem: P, Epoch: 4, Current best: -1.90625, Global best: -1.90625, Runtime: 0.50307 seconds
2024/07/10 03:49:07 PM, INFO, mealpy.bio_based.VCS.DevVCS: >>>Problem: P, Epoch: 5, Current best: -1.90625, Global best: -1.90625, Runtime: 0.51970 seconds
2024/07/10 03:49:08 PM, INFO, mealpy.bio_based.VCS.DevVCS: >>>Problem: P, Epoch: 6, Current best: -2.375, Global b

KeyboardInterrupt: 

In [61]:
from mealpy import FloatVar, WHO

problem_dict = {
    "bounds": FloatVar(lb=(-100.,) * dimension, ub=(100.,) * dimension, name="delta"),
    "minmax": "min",
    "obj_func": evaluate_expectation
}

model = WHO.OriginalWHO(epoch=1000, pop_size=50, n_explore_step = 3, n_exploit_step = 3, eta = 0.15, p_hi = 0.9,
                        local_alpha=0.9, local_beta=0.3, global_alpha=0.2, global_beta=0.8, delta_w=2.0, delta_c=2.0)
g_best = model.solve(problem_dict)
print(f"Solution: {g_best.solution}, Fitness: {g_best.target.fitness}")
print(f"Solution: {model.g_best.solution}, Fitness: {model.g_best.target.fitness}")

2024/07/10 03:52:52 PM, INFO, mealpy.bio_based.WHO.OriginalWHO: Solving single objective optimization problem.
2024/07/10 03:52:54 PM, INFO, mealpy.bio_based.WHO.OriginalWHO: >>>Problem: P, Epoch: 1, Current best: -1.71875, Global best: -1.71875, Runtime: 1.38588 seconds
2024/07/10 03:52:55 PM, INFO, mealpy.bio_based.WHO.OriginalWHO: >>>Problem: P, Epoch: 2, Current best: -1.71875, Global best: -1.71875, Runtime: 1.21608 seconds
2024/07/10 03:52:56 PM, INFO, mealpy.bio_based.WHO.OriginalWHO: >>>Problem: P, Epoch: 3, Current best: -1.71875, Global best: -1.71875, Runtime: 1.30477 seconds
2024/07/10 03:52:58 PM, INFO, mealpy.bio_based.WHO.OriginalWHO: >>>Problem: P, Epoch: 4, Current best: -1.71875, Global best: -1.71875, Runtime: 1.42247 seconds
2024/07/10 03:52:59 PM, INFO, mealpy.bio_based.WHO.OriginalWHO: >>>Problem: P, Epoch: 5, Current best: -1.875, Global best: -1.875, Runtime: 1.36160 seconds
2024/07/10 03:53:00 PM, INFO, mealpy.bio_based.WHO.OriginalWHO: >>>Problem: P, Epoch: 6,

KeyboardInterrupt: 